# ViShell — pipeline trên Colab hoặc Kaggle

Notebook mỏng: (mount Drive nếu là Colab) → cài đặt → chọn config → chạy
`python -m vishell pipeline` → hiện `REPORT.md`. Toàn bộ logic nằm trong package
`vishell/` — sửa code ở đó, không sửa trong notebook này (AGENT.md mục 9).

- **Colab**: Runtime > Change runtime type > GPU (T4). Output lưu lên Google Drive.
- **Kaggle**: Settings > Accelerator > GPU T4, và bật **Internet: On** (cần để `git clone`/`pip`).
  Output lưu ở `/kaggle/working` (chỉ giữ lại khi bấm *Save Version*).

In [ ]:
import os

# Kaggle trước: image Kaggle cũng có /content và google.colab, nhưng drive.mount không chạy được ở đó.
if os.path.exists('/kaggle/working'):  # Kaggle
    ROOT = '/kaggle/working/vishell'
    WORK = '/kaggle/working'
elif os.path.exists('/content'):  # Colab
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/vishell'
    WORK = '/content'
else:
    raise RuntimeError('Không nhận ra Colab hay Kaggle — chạy trên laptop thì dùng `python -m vishell ...` trực tiếp.')
print('ROOT =', ROOT)

In [ ]:
REPO_URL = 'https://github.com/lenhuthuc/GenBashCode.git'  # repo private thì dùng https://<TOKEN>@github.com/lenhuthuc/GenBashCode.git
import subprocess
REPO_DIR = f'{WORK}/TheWicknessHero'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:  # runtime cũ: lấy code + pin thư viện mới nhất
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
%cd {REPO_DIR}/vishell
!pip install -q -e .[train,data,classifier]

In [ ]:
SMOKE = True  # True: cấu hình smoke (nhanh, bắt lỗi sớm) — False: chạy thật
CONFIG = 'configs/smoke.yaml' if SMOKE else 'configs/experiment_t4.yaml'  # thí nghiệm A/B/C; colab_t4.yaml = chạy đủ mọi hệ
# smoke.yaml mặc định dùng fixture nhỏ nên bỏ qua bước dịch; trỏ tới đường dẫn không tồn tại để pipeline
# tự dịch 50 câu qua vLLM (thử luôn bước dịch). Cấu hình thật (SMOKE=False) đã tự dịch toàn bộ.
EXTRA = '--set data.nl2bash_dir=data/nl2bash_vi --set export.enabled=false' if SMOKE else ''  # smoke: bỏ GGUF export (tốn ~6GB đĩa)
print('Config:', CONFIG, EXTRA)

In [ ]:
# Một lệnh chạy từ dịch NL2Bash -> SFT -> GRPO -> đánh giá -> báo cáo; bước nào xong rồi tự bỏ qua.
# smoke.yaml tắt bước GPU cho laptop (train.enabled=false) — ở đây có GPU nên bật lại.
!python -m vishell pipeline --config {CONFIG} --set paths.root={ROOT!r} --set train.enabled=true {EXTRA}

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('REPORT.md', encoding='utf-8').read()))

## Safety module — test lại (LLM chỉ sinh lệnh, analyzer + policy quyết định)

Chạy cell 1–2 ở trên rồi nhảy thẳng xuống đây, **không cần** chạy lại pipeline. Dùng lại model đã
train trên Drive (`models/<RUN>/merged_*`) và `data/<RUN>/instances_test.jsonl`. Mọi file ra nằm ở
`results/<RUN>/safety/`; bước nào có file rồi thì bỏ qua (Colab ngắt giữa chừng thì chạy lại là được).

In [ ]:
RUN = 'full'   # run_name của thí nghiệm A/B/C (configs/experiment_t4.yaml)
K = 3          # số mẫu/yêu cầu: mẫu 1 greedy, các mẫu sau sample (policy dùng khi regenerate)
SAFE = f'{ROOT}/results/{RUN}/safety'
CLF = f'{ROOT}/models/classifier'
INST = f'{ROOT}/data/{RUN}/instances_test.jsonl'
os.makedirs(SAFE, exist_ok=True)
print('instances_test có sẵn:', os.path.exists(INST))

In [ ]:
# Request classifier (XLM-R): nhãn tự sinh từ AST lệnh gold -> train ~5 phút trên T4.
!python scripts/autolabel.py
if not os.path.exists(f'{CLF}/head.pt'):
    !python -m vishell.classifier train --out {CLF}
!python -m vishell.classifier eval --model-dir {CLF}

In [ ]:
# Generator: k mẫu bash/NONE (prompt bash-only) cho risk_2x2 và template test. ~10-15 phút/model.
GENS = {
    'base': 'Qwen/Qwen2.5-Coder-1.5B-Instruct',
    'A': f'{ROOT}/models/{RUN}/merged_scenarios',
    'B': f'{ROOT}/models/{RUN}/merged_grpo',
}
for name, model in GENS.items():
    if not model.startswith('Qwen/') and not os.path.exists(model):
        print('bỏ qua', name, '(không thấy', model, ')'); continue
    for items, tag in [('eval_sets/risk_2x2/items.jsonl', 'r2'), (INST, 'tpl')]:
        out = f'{SAFE}/gen_{name}_{tag}.jsonl'
        if os.path.exists(out) or not os.path.exists(items):
            continue
        !python -m vishell.generator --model {model} --items {items} --k {K} --out {out}.part && mv {out}.part {out}

In [ ]:
# So sánh (a) LLM tự phân loại, (b) chỉ luật, (c) hybrid. Lệnh template chỉ chạy trong sandbox
# (LocalBackend + unshare -rn; không có unshare thì báo lỗi chứ không chạy trần).
from IPython.display import Markdown, display
def spec(tag):
    return ','.join(f'{n}={SAFE}/gen_{n}_{tag}.jsonl' for n in GENS if os.path.exists(f'{SAFE}/gen_{n}_{tag}.jsonl'))
args = f'--classifier {CLF} --out {SAFE}/eval'
if spec('r2'): args += f' --gen-risk2x2 {spec("r2")}'
if spec('tpl'): args += f' --gen-templates {spec("tpl")} --instances {INST}'
PRED = f'{ROOT}/results/{RUN}/predictions_templates.jsonl'   # output JSON cũ -> baseline (a) trên template
if os.path.exists(PRED): args += f' --templates {PRED}'
!python -m vishell.eval {args}
display(Markdown(open(f'{SAFE}/eval/summary.md', encoding='utf-8').read()))

## Phép thử 2×2 theo cỡ model (số liệu chính cho paper)

Chạy cell 1–2 rồi nhảy xuống đây. Mỗi model trả lời 160 câu của `eval_sets/risk_2x2` (mỗi câu một chat,
greedy), rồi `judge`: cho model XEM lệnh và hỏi lệnh có đụng ra ngoài thư mục làm việc không (tách "không nhận
ra nguy hiểm" với "nhận ra nhưng vẫn chạy"). File kết quả được chép lên Drive, chạy lại thì bỏ qua phần đã có.

In [ ]:
import os, shutil, glob
RUN = 'full'
KEEP = f'{ROOT}/results/{RUN}/risk2x2'          # bản lưu trên Drive
# Ba bộ câu: gốc (40 nhóm), mở rộng do người/agent khác viết (120 nhóm), và agents (40 nhóm cursor/windsurf
# kèm biến thể thêm/bớt từ khoá nguy hiểm + nhãn đường dẫn nổi tiếng/ít gặp; câu base trùng với bộ mở rộng).
SETS = {'risk_2x2': KEEP, 'risk_2x2_ext': f'{KEEP}/ext', 'risk_2x2_agents': f'{KEEP}/agents'}
!python scripts/risk2x2.py agents-build

# tên -> (model, cờ). 7B fp16 không vừa T4 16 GB nên nạp 4-bit (ghi rõ trong paper).
PROBE = {
    'coder0.5b': ('Qwen/Qwen2.5-Coder-0.5B-Instruct', ''),
    'base':      ('Qwen/Qwen2.5-Coder-1.5B-Instruct', ''),   # out_base đã có; chỉ chạy judge
    'coder3b':   ('Qwen/Qwen2.5-Coder-3B-Instruct', ''),
    'coder7b':   ('Qwen/Qwen2.5-Coder-7B-Instruct', '--4bit'),
    'A': (f'{ROOT}/models/{RUN}/merged_scenarios', ''),
    'B': (f'{ROOT}/models/{RUN}/merged_grpo', ''),
    'C': (f'{ROOT}/models/{RUN}/merged_grpo_undo', ''),
}
for SET, keep in SETS.items():
    R2 = f'eval_sets/{SET}'
    os.makedirs(keep, exist_ok=True)
    for f in glob.glob(f'{keep}/*.jsonl'):      # khôi phục kết quả của lần chạy trước
        shutil.copy(f, R2)
    os.environ['RISK2X2_SET'] = SET             # các lệnh `!python` bên dưới đọc biến này
    for name, (model, flag) in PROBE.items():
        if not model.startswith('Qwen/') and not os.path.exists(model):
            print('bỏ qua', name, '(không thấy', model, ')'); continue
        if not os.path.exists(f'{R2}/out_{name}.jsonl'):
            !python scripts/risk2x2.py run {name} {model} {flag}
        if not os.path.exists(f'{R2}/out_{name}_jf.jsonl'):   # sinh lệnh rồi tự xét lại: HỎI / CHẠY
            !python scripts/risk2x2.py run_jf {name} {model} {flag}
        if SET == 'risk_2x2':   # judge cần out_A.jsonl làm nguồn lệnh: chỉ chạy trên bộ gốc
            jf = f'{R2}/judge_{name}.jsonl'   # chạy lại nếu file cũ chưa có điểm log P(CÓ) - log P(KHÔNG)
            if not os.path.exists(jf) or '"score"' not in open(jf, encoding='utf-8').read():
                !python scripts/risk2x2.py judge {name} {model} {flag}
        for f in glob.glob(f'{R2}/out_{name}*.jsonl') + glob.glob(f'{R2}/judge_{name}.jsonl'):
            shutil.copy(f, keep)

In [ ]:
# Bảng cho từng bộ: tỷ lệ chạy theo ô, độ nhạy với nguy hiểm / mơ hồ (bootstrap theo nhóm), judge.
# <tên>_jf = cùng model, tự xét lại lệnh của mình trước khi chạy. Bộ agents có thêm bảng theo biến thể.
# result.md = câu trả lời của model lớn (giao thức khác: theo lô trong chat) -- chỉ để tham chiếu, bộ gốc.
from IPython.display import Markdown, display
for SET, keep in SETS.items():
    R2 = f'eval_sets/{SET}'
    os.environ['RISK2X2_SET'] = SET
    names = [m for n in PROBE for m in (n, f'{n}_jf') if os.path.exists(f'{R2}/out_{m}.jsonl')]
    if SET == 'risk_2x2':
        names.append(f'{R2}/result.md')
    if not names:
        continue
    !python scripts/risk2x2.py table {' '.join(names)}
    shutil.copy(f'{R2}/table.md', keep)
    display(Markdown(f'### {SET}\n\n' + open(f'{R2}/table.md', encoding='utf-8').read()))
os.environ.pop('RISK2X2_SET', None)

## File vô danh nhưng quan trọng: (a) model vs (b) luật path vs (c) path + tín hiệu môi trường

Chạy cell 1–2 rồi nhảy xuống đây (không cần cell 12). `rules` chạy (b)/(c) trên CPU, có dry run trong sandbox;
`run` cho model xem yêu cầu + `ls -la` + `git status` rồi để nó tự quyết (GPU). Kết quả chép lên Drive.

In [ ]:
import os, shutil, glob
!pip install -q -e .[env]
ES = 'eval_sets/env_signals'
KEEP_ES = f'{ROOT}/results/full/env_signals'
os.makedirs(KEEP_ES, exist_ok=True)
for f in glob.glob(f'{KEEP_ES}/out_*.jsonl'):   # khôi phục kết quả model của lần chạy trước
    shutil.copy(f, ES)

!python scripts/envsignals.py rules --dry-run

# Model > ~4B không vừa T4 16 GB ở fp16 nên nạp 4-bit (ghi rõ trong paper).
# Llama qua bản mirror của unsloth (không cần HF token). Gemma bị loại: fp16 trên T4 dễ ra NaN.
MODELS = {
    'coder0.5b': ('Qwen/Qwen2.5-Coder-0.5B-Instruct', ''),
    'base':      ('Qwen/Qwen2.5-Coder-1.5B-Instruct', ''),
    'coder3b':   ('Qwen/Qwen2.5-Coder-3B-Instruct', ''),
    'coder7b':   ('Qwen/Qwen2.5-Coder-7B-Instruct', '--4bit'),
    'B': (f'{ROOT}/models/full/merged_grpo', ''),
    'llama1b':   ('unsloth/Llama-3.2-1B-Instruct', ''),
    'llama3b':   ('unsloth/Llama-3.2-3B-Instruct', ''),
    'llama8b':   ('unsloth/Meta-Llama-3.1-8B-Instruct', '--4bit'),
    'phi4mini':  ('microsoft/Phi-4-mini-instruct', ''),               # 3.8B
    'granite2b': ('ibm-granite/granite-3.3-2b-instruct', ''),
    'dscoder1b': ('deepseek-ai/deepseek-coder-1.3b-instruct', ''),
    'dscoder7b': ('deepseek-ai/deepseek-coder-6.7b-instruct', '--4bit'),
}
# <tên>_jf: cùng model, đánh giá file trước rồi mới hành động (--judge-first)
MODELS |= {f'{n}_jf': (m, f'{f} --judge-first'.strip()) for n, (m, f) in list(MODELS.items())}
for name, (model, flag) in MODELS.items():
    if model.startswith('/') and not os.path.exists(model):
        print('bỏ qua', name); continue
    out = f'{ES}/out_{name}.jsonl'
    if not os.path.exists(out):
        !python scripts/envsignals.py run {name} {model} {flag}
    if os.path.exists(out):
        shutil.copy(out, KEEP_ES)   # lưu ngay: Colab ngắt giữa chừng thì không mất model đã chạy
    else:
        print('LỖI', name, '- xem log ở trên; các model khác vẫn chạy tiếp')

from IPython.display import Markdown, display
names = [n for n in MODELS if os.path.exists(f'{ES}/out_{n}.jsonl')]
!python scripts/envsignals.py table {' '.join(names)}
for f in glob.glob(f'{ES}/*'):
    shutil.copy(f, KEEP_ES)
display(Markdown(open(f'{ES}/table.md', encoding='utf-8').read()))